# Initialization

In [1]:
# Initialize current working directory
import os

os.chdir(os.path.dirname(os.getcwd()))
os.getcwd()

'/Users/gurveersinghvirk/Documents/github/CodeSage'

In [2]:
# Initialize COMStore
from backend.code_sage.com_store import COMStore
from backend.config import config

com_store = COMStore(
    url=config.NEO4J_URI,
    username=config.NEO4J_USER,
    password=config.NEO4J_PASSWORD,
)

In [3]:
from usage_validation.main import find_external_entities, validate_code, generate_stub_directory

# Test custom stub generation

In [5]:
es = {
  "aerosandbox.Wing": {
    "type": "CLASS",
    "signature": "(self, name: Optional[str] = None, xsecs: List[ForwardRef('WingXSec')] = None, symmetric: bool = False, color: Union[str, Tuple[float], NoneType] = None, analysis_specific_options: Optional[Dict[type, Dict[str, Any]]] = None, **kwargs)"
  },
  "aerosandbox.Airfoil": {
    "type": "CLASS",
    "signature": "(self, name: str = 'Untitled', coordinates: Union[NoneType, str, pathlib.Path, numpy.ndarray] = None, **deprecated_keyword_arguments)"
  },
  "aerosandbox.LiftingLine": {
    "type": "CLASS",
    "signature": "(self, airplane: aerosandbox.geometry.airplane.Airplane, op_point: aerosandbox.performance.operating_point.OperatingPoint, xyz_ref: List[float] = None, model_size: str = 'medium', run_symmetric_if_possible: bool = False, verbose: bool = False, spanwise_resolution: int = 4, spanwise_spacing_function: Callable[[float, float, float], numpy.ndarray] = <function cosspace at 0x0000013F0B3B6160>, vortex_core_radius: float = 1e-08, align_trailing_vortices_with_wind: bool = False)"
  },
  "aerosandbox.Airplane": {
    "type": "CLASS",
    "signature": "(self, name: Optional[str] = None, xyz_ref: Union[numpy.ndarray, List] = None, wings: Optional[List[aerosandbox.geometry.wing.Wing]] = None, fuselages: Optional[List[aerosandbox.geometry.fuselage.Fuselage]] = None, propulsors: Optional[List[aerosandbox.geometry.propulsor.Propulsor]] = None, s_ref: Optional[float] = None, c_ref: Optional[float] = None, b_ref: Optional[float] = None, analysis_specific_options: Optional[Dict[type, Dict[str, Any]]] = None)"
  },
  "aerosandbox.LiftingLine.run": {
    "type": "METHOD",
    "signature": "(self) -> Dict"
  },
  "aerosandbox.OperatingPoint": {
    "type": "CLASS",
    "signature": "(self, atmosphere: aerosandbox.atmosphere.atmosphere.Atmosphere = Atmosphere (altitude: 0 m (0 ft), method: 'differentiable'), velocity: float = 1.0, alpha: float = 0.0, beta: float = 0.0, p: float = 0.0, q: float = 0.0, r: float = 0.0)"
  },
  "aerosandbox.WingXSec": {
    "type": "CLASS",
    "signature": "(self, xyz_le: Union[numpy.ndarray, List] = None, chord: float = 1.0, twist: float = 0.0, airfoil: aerosandbox.geometry.airfoil.airfoil.Airfoil = None, control_surfaces: Optional[List[ForwardRef('ControlSurface')]] = None, analysis_specific_options: Optional[Dict[type, Dict[str, Any]]] = None, **deprecated_kwargs)"
  }
}

In [6]:
generate_stub_directory(entities=es, root_stub_dir="/Users/gurveersinghvirk/Documents/github/CodeSage/usage_validation")

# Usage Validation Gradio App

Select test files from `usage_validation/tests` and run `check_usages` to validate usages of functions, classes, etc.

Note: `check_usages` is expected to be provided elsewhere; this notebook only wires up the UI.

In [15]:
# Setup: imports and path constants
from pathlib import Path
from typing import List, Dict

# Base directory for tests
PROJECT_ROOT = Path("/Users/gurveersinghvirk/Documents/github/CodeSage").resolve()
TESTS_DIR = PROJECT_ROOT / "usage_validation" / "tests"
TEST_CASES_DIR = TESTS_DIR / "test_cases"
OUTPUTS_DIR = TESTS_DIR / "outputs"

# Helper to list available files (relative to tests dir)
def list_test_files() -> List[str]:
    files = []
    if TESTS_DIR.exists():
        for p in sorted(TESTS_DIR.rglob("*.py")):
            if p.is_file() and not p.name.startswith('.'):
                files.append(str(p.relative_to(TESTS_DIR)))
    return files

# Helper to get signatures from COMStore
def get_signatures(entities: List[str]) -> Dict[str, str]:
    """Fetches signatures for a list of entities from the COM store."""
    if not entities:
        print("No entities provided for signature retrieval.")
        return {}
    return com_store.get_signatures(entities)

# Main validation function
def check_usages(selected_files: List[str]) -> Dict:
    """
    Runs usage validation on a list of selected files.
    """
    results = {}
    for file_rel_path in selected_files:
        try:
            file_abs_path = TESTS_DIR / file_rel_path
            code_content = file_abs_path.read_text(encoding='utf-8')

            # 1. Find external entities in the code.
            entities_in_code = find_external_entities(code_content)

            # 2. Fetch signatures for those entities.
            signatures_to_use = get_signatures(list(entities_in_code))

            # 3. Run the complete validation.
            validation_result = validate_code(code_content, signatures_to_use)
            
            results[file_rel_path] = validation_result

        except Exception as e:
            results[file_rel_path] = {
                "error": f"Failed to process file: {type(e).__name__}: {e}"
            }
    return results

files_list_cache = list_test_files()
print(f"Discovered {len(files_list_cache)} files under {TESTS_DIR}")

Discovered 8 files under /Users/gurveersinghvirk/Documents/github/CodeSage/usage_validation/tests


In [24]:
# Gradio UI (with Select All / None)
import gradio as gr
from typing import List

# Ensure choices are current
CHOICES = list_test_files() if 'list_test_files' in globals() else []

with gr.Blocks(title="Usage Validation") as demo:
    gr.Markdown(
        """
        ## Validate Usages (v2)
        - Select files from `usage_validation/tests`.
        - Use Select All / None to quickly adjust the selection.
        - Click Run to execute `check_usages` on the selection.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Row():
                refresh_btn = gr.Button("Refresh list", variant="secondary")
                select_all_btn = gr.Button("Select all")
                select_none_btn = gr.Button("Select none")
            file_selector = gr.CheckboxGroup(
                label="Test files",
                choices=CHOICES,
                value=[],
                info="Paths are relative to usage_validation/tests"
            )
        with gr.Column(scale=1):
            run_btn = gr.Button("Run validation", variant="primary")
            clear_btn = gr.Button("Clear output", variant="secondary")
            download_all_btn = gr.Button("Download all")

    results_state = gr.State({})
    output_text = gr.Textbox(label="Log / Errors", lines=5)
    files_download = gr.Files(label="Generated .txt outputs", file_count="multiple")
    
    with gr.Row():
        result_selector = gr.Dropdown(label="Select a file to view details", choices=[], scale=1)
        selected_result_output = gr.Textbox(label="Selected Result Details", lines=20, scale=2, interactive=True, show_copy_button=True, show_label=True)

    # Actions
    def do_refresh():
        global files_list_cache
        files_list_cache = list_test_files()
        return gr.update(choices=files_list_cache, value=[])

    def select_all() -> List[str] | dict:
        # Set selection to all available choices
        current = list_test_files()
        return gr.update(value=current)

    def select_none() -> List[str] | dict:
        return gr.update(value=[])

    def run_validation(selected: List[str]):
        try:
            result = check_usages(selected)
            if isinstance(result, dict):
                file_keys = list(result.keys())
                log_message = f"Validation completed for {len(file_keys)} files."
                first_key = file_keys[0] if file_keys else None
                
                # Initial detail view
                initial_details = result.get(first_key, {}) if first_key else ""

                return log_message, result, gr.update(choices=file_keys, value=first_key), initial_details
            
            return str(result), {}, gr.update(choices=[], value=None), ""

        except NotImplementedError as nie:
            return f"Not implemented: {nie}", {}, gr.update(choices=[], value=None), ""
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}", {}, gr.update(choices=[], value=None), ""

    def display_selected_result(selection, all_results):
        if not selection or not isinstance(all_results, dict):
            return ""
        selected_data = all_results.get(selection, "")
        return selected_data

    def clear_outputs():
        return "", {}, gr.update(choices=[], value=None), ""

    def download_all(selected: List[str], all_results: dict):
        from pathlib import Path
        try:
            out_paths: List[str] = []
            OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
            if not selected:
                return []
            if not isinstance(all_results, dict):
                all_results = {}
            for rel in selected:
                data = all_results.get(rel, None)
                # Build destination under outputs/, mirroring subdirs, .py -> .txt
                rel_path = Path(rel)
                # Change suffix to .txt
                dest_rel = rel_path.with_suffix(".txt")
                # Delete test_cases/ from the path
                dest_rel = Path(str(dest_rel).replace("test_cases/", ""))
                dest_path = OUTPUTS_DIR / dest_rel
                dest_path.parent.mkdir(parents=True, exist_ok=True)
                dest_path.write_text(data, encoding="utf-8")
                out_paths.append(str(dest_path))
            return out_paths
        except Exception as e:
            print(f"Download all error: {type(e).__name__}: {e}")
            return []

    # Wiring
    refresh_btn.click(fn=do_refresh, outputs=file_selector)
    select_all_btn.click(fn=select_all, outputs=file_selector)
    select_none_btn.click(fn=select_none, outputs=file_selector)
    
    run_btn.click(
        fn=run_validation, 
        inputs=file_selector, 
        outputs=[output_text, results_state, result_selector, selected_result_output]
    )
    
    result_selector.change(
        fn=display_selected_result,
        inputs=[result_selector, results_state],
        outputs=selected_result_output
    )

    clear_btn.click(
        fn=clear_outputs, 
        outputs=[output_text, results_state, result_selector, selected_result_output]
    )

    download_all_btn.click(
        fn=download_all,
        inputs=[file_selector, results_state],
        outputs=files_download
    )

INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


In [25]:
# Launch the Gradio app inline (run this cell)
demo.launch(share=False, inline=False)

INFO:httpx:HTTP Request: GET http://127.0.0.1:7868/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7868/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
